In [1]:
!pip install -q groq gradio yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 2.0 MB/s eta 0:00:00


In [2]:
import os

os.environ["GROQ_API_KEY"] = "YOUR_API_KEY"

In [3]:
from groq import Groq
import os

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [13]:
def financial_assistant(user_input):
    try:
        prompt = f"""
You are a professional financial analyst.

Answer ONLY finance-related questions.

Format:
- Summary
- Key Points
- Risks

If the question is not finance-related, say:
"I can only answer finance-related questions."

Question: {user_input}
"""

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"Error: {str(e)}"

In [14]:
import yfinance as yf

def get_stock_price(ticker):
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period="1d")
        if not data.empty:
            return round(data["Close"].iloc[-1], 2)
        return None
    except:
        return None

In [15]:
def smart_assistant(user_input):
    words = user_input.split()

    for word in words:
        if word.isupper() and len(word) <= 5:
            price = get_stock_price(word)
            if price:
                user_input += f"\nCurrent price of {word} is {price}"

    return financial_assistant(user_input)

In [20]:
import gradio as gr

# ---------- PRESET FUNCTIONS ----------
def beginner_advice():
    return smart_assistant("I am a beginner with 10000 INR. How should I start investing?")

def analyze_stock():
    return smart_assistant("Analyze AAPL stock and explain risks")

def risk_analysis():
    return smart_assistant("Explain risk management in investing")

# ---------- UI ----------
with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown("#  AI Financial Assistant")
    gr.Markdown("### Get smart financial insights with real-time stock data ")

    with gr.Row():
        with gr.Column(scale=2):
            user_input = gr.Textbox(
                lines=6,
                placeholder="Ask anything about stocks, investing, risks...",
                label=" Your Question"
            )

            submit_btn = gr.Button(" Analyze", variant="primary")

        with gr.Column(scale=3):
            output = gr.Textbox(
                lines=18,
                label=" AI Financial Insights"
            )

    # ---------- BUTTON ACTION ----------
    submit_btn.click(fn=smart_assistant, inputs=user_input, outputs=output)

    # ---------- QUICK ACTION BUTTONS ----------
    gr.Markdown("##  Quick Actions")

    with gr.Row():
        btn1 = gr.Button(" Analyze Apple (AAPL)")
        btn2 = gr.Button(" Beginner Investment Advice")
        btn3 = gr.Button(" Risk Management")

    btn1.click(fn=analyze_stock, outputs=output)
    btn2.click(fn=beginner_advice, outputs=output)
    btn3.click(fn=risk_analysis, outputs=output)

    # ---------- FOOTER ----------
    gr.Markdown("---")
    gr.Markdown("Built with using LLMs + Real-Time Data")

app.launch()

/tmp/ipykernel_515/4137381512.py:14: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://92e2f398a3c3a0d33e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
